# Matching
Now that we have the .csv with the speeches, last name and first name of the deputies, we have to match the names with the information we have on the database

In [1]:
import pandas as pd
import sqlite3

In [2]:
LOCATION_SPEECHES = ("../../data/processed/speeches.csv")
LOCATION_DB = ("../../data/chilean_polispeech.db")

# save output
SAVE_RESULTS = ("../../data/processed/matched_speeches.csv")

In [3]:
clean_speeches = pd.read_csv(LOCATION_SPEECHES)
clean_speeches

,law_number,law_title,section_key,date,speaker_header_raw,speaker_article,speaker_body_raw,speech_raw,speech_clean,turn_type
0,21.472,Crea un Fondo de Estabilización de Tarifas y e...,1-5,2022-06-07,"El señor SOTO, don Raúl (Presidente).-",El,"SOTO, don Raúl (Presidente)","Iniciando el Orden del Día, corresponde tratar...",NaN,floor_control
1,21.472,Crea un Fondo de Estabilización de Tarifas y e...,1-5,2022-06-07,"El señor SOTO, don Raúl (Presidente).-",El,"SOTO, don Raúl (Presidente)","Doy la bienvenida al ministro de Energía, seño...",NaN,floor_control
2,21.472,Crea un Fondo de Estabilización de Tarifas y e...,1-5,2022-06-07,"La señorita AHUMADA, doña Yovana (de pie).-",La,"AHUMADA, doña Yovana (de pie)","Señor Presidente, en mi calidad de diputada in...",NaN,law_introduction
3,21.472,Crea un Fondo de Estabilización de Tarifas y e...,1-5,2022-06-07,"El señor SOTO, don Raúl (Presidente).-",El,"SOTO, don Raúl (Presidente)",Tiene la palabra el diputado informante de la ...,NaN,floor_control
4,21.472,Crea un Fondo de Estabilización de Tarifas y e...,1-5,2022-06-07,El señor NARANJO (de pie).-,El,NARANJO (de pie),"Señor Presidente, honorable Cámara, en nombre ...",NaN,floor_control
...,...,...,...,...,...,...,...,...,...,...
2476,21.800,Regula la extracción de áridos y modifica los ...,2-5,2024-12-17,El señor AEDO (Vicepresidente).-,El,AEDO (Vicepresidente),Corresponde votar en particular el artículo 21...,NaN,floor_control
2477,21.800,Regula la extracción de áridos y modifica los ...,2-5,2024-12-17,El señor AEDO (Vicepresidente).-,El,AEDO (Vicepresidente),Aprobado.\n-Votaron por la afirmativa:\nAceved...,NaN,floor_control
2478,21.800,Regula la extracción de áridos y modifica los ...,2-5,2024-12-17,El señor AEDO (Vicepresidente).-,El,AEDO (Vicepresidente),Corresponde votar en particular el artículo te...,NaN,floor_control
2479,21.800,Regula la extracción de áridos y modifica los ...,2-5,2024-12-17,El señor AEDO (Vicepresidente).-,El,AEDO (Vicepresidente),Aprobado.\n-Votaron por la afirmativa:\nAceved...,NaN,floor_control


In [32]:
# filter by the speeches we care about
filtered_speakers = clean_speeches.loc[clean_speeches["turn_type"] == "substantive"].copy()

# since we are sure now that we got only the substantive speeches, we c an drop the columns that we used to classifgy the speeches
filtered_speakers = filtered_speakers.drop(
    columns=["speaker_header_raw", "speaker_article", "turn_type"]
)


### Extract data from speaker_body_raw

In [33]:
# extract the speaker body for name extraction
speaker_body = filtered_speakers["speaker_body_raw"]

# we obtain the last name by looking for all uppercase words and joining them together, then lowercasing the result
filtered_speakers["lastname1"] = (
    speaker_body.str.findall(r"\b[A-ZÁÉÍÓÚÑÜ]+(?:-[A-ZÁÉÍÓÚÑÜ]+)?\b") # returns a list
    .str.join(" ") # in case of composed last names (e.g. "De la Carrera"), it joins them together with a space
    .str.lower()
)

# we obtain the first name by looking for the words following "don" or "doña", and joining them together, then lowercasing the result
filtered_speakers["firstname"] = speaker_body.str.extract( 
    r"(?i)\b(?:don|doña)\b\s*([^\(\)]+?)(?=\s*[\(\)]|\s*$)" # returns a DataFrame
)[0].str.lower()

#filtered_speakers



### Load deputies from SQLite

In [34]:
con = sqlite3.connect(LOCATION_DB)
cur = con.cursor()

deputies = pd.read_sql_query("""
    SELECT dipid, name, lastname_1, lastname_2, sex_value
    FROM deputies""",
    con)

con.close()

In [35]:
#Prepare both tables for matching

# Lowercase + strip only for matching
# We keep the original columns untouched, so the matched result still shows normal names
deputies["lastname_1_l"] = deputies["lastname_1"].str.lower().str.strip()
deputies["name_l"] = deputies["name"].str.lower().str.strip()

filtered_speakers["lastname_1_l"] = filtered_speakers["lastname1"].str.lower().str.strip()

# firstname has missing values, so fill them with ""
filtered_speakers["firstname_l"] = (
    filtered_speakers["firstname"]
    .fillna("")
    .str.lower()
    .str.strip()
)

# Take only the first token of the given name
deputies["first_name_token"] = deputies["name_l"].str.split().str[0]
filtered_speakers["first_name_token"] = filtered_speakers["firstname_l"].str.split().str[0]

### Matching
It will be done in two different passes. Because from the header of when deputies speak we find that there are some that appear with Lastname (Surname) and some with Lastname and Firstname (Given name). This is because the first name is only used when the Lastname of the deputy is not unique.

In [36]:
# First attempt to match: Lastname + first-name token

final_match = filtered_speakers.merge(
    deputies[
        [   "dipid",
            "lastname_1_l",
            "first_name_token",
            "name",
            "lastname_1",
            "lastname_2",
            "sex_value"]
    ].rename(columns={
        "name": "matched_name",
        "lastname_1": "matched_lastname_1",
        "lastname_2": "matched_lastname_2",
        "sex_value": "matched_sex_value"
    }),
    on=["lastname_1_l", "first_name_token"], # match by last name and first name token
    how="left"
)

# Save which rows already matched in the first pass
final_match["matched_1pass"] = final_match["dipid"].notna()
print("Matched first pass:", final_match["dipid"].notna().sum())
print("Still unmatched:", final_match["dipid"].isna().sum())

Matched first pass: 557
Still unmatched: 554


In [37]:
# Second attempt: surname only (unique)

# Count how many times each surname appears in the deputies table
surname_counts = deputies["lastname_1_l"].value_counts()

# Keep only deputies whose first surname appears exactly once
deputies_once = deputies.loc[
    deputies["lastname_1_l"].map(surname_counts) == 1,
    ["lastname_1_l", "dipid", "name", "lastname_1", "lastname_2", "sex_value"]
].drop_duplicates(subset="lastname_1_l").set_index("lastname_1_l")

# Mask = rows still unmatched after the first merge
mask = final_match["dipid"].isna()

# Fill only those unmatched rows using surname-only lookup
final_match.loc[mask, "dipid"] = final_match.loc[mask, "lastname_1_l"].map(deputies_once["dipid"])
final_match.loc[mask, "matched_name"] = final_match.loc[mask, "lastname_1_l"].map(deputies_once["name"])
final_match.loc[mask, "matched_lastname_1"] = final_match.loc[mask, "lastname_1_l"].map(deputies_once["lastname_1"])
final_match.loc[mask, "matched_lastname_2"] = final_match.loc[mask, "lastname_1_l"].map(deputies_once["lastname_2"])
final_match.loc[mask, "matched_sex_value"] = final_match.loc[mask, "lastname_1_l"].map(deputies_once["sex_value"])


In [38]:
# --- 4) Label how each row was matched ---

final_match["match_status"] = "unmatched"

# Matched using surname + first name
final_match.loc[
    final_match["matched_1pass"],"match_status"] = "matched_name_lastname"

# Matched using surname only
final_match.loc[
    (~final_match["matched_1pass"]) & final_match["dipid"].notna(),
    "match_status"
] = "matched_lastname_only"

# Count the result
final_match["match_status"].value_counts(dropna=False)

match_status
matched_name_lastname    557
matched_lastname_only    551
unmatched                  3
Name: count, dtype: int64

There are some edge cases, but the error are in the transcription itself. They will be explained in the written report.

## Fill SQL


In [40]:
con = sqlite3.connect(LOCATION_DB)
cur = con.cursor()

In [41]:
cur.execute("""
CREATE TABLE IF NOT EXISTS law_sections (
    law_section_id INTEGER PRIMARY KEY AUTOINCREMENT,
    law_number TEXT NOT NULL,
    section_key TEXT NOT NULL,
    discussion_date TEXT,
    law_title TEXT,
    UNIQUE (law_number, section_key)
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS speeches (
    speech_id INTEGER PRIMARY KEY AUTOINCREMENT,
    law_section_id INTEGER NOT NULL,
    dipid INTEGER,
    speech_raw TEXT,
    speech_clean TEXT NOT NULL,
    FOREIGN KEY (law_section_id) REFERENCES law_sections (law_section_id),
    FOREIGN KEY (dipid) REFERENCES deputies (dipid)
);
""")

con.commit()

In [42]:
# create a copy of the matched speeches
matched_speeches = final_match.copy()
# convert the law_number to string
matched_speeches["law_number"] = matched_speeches["law_number"].astype(str)

In [43]:
# here we create the law_sections table by taking the unique combinations of law_number, section_key, date and law_title from the matched speeches

# for security
if "law_title" not in matched_speeches.columns:
    matched_speeches["law_title"] = None

# only one row per law_number + section_key + date +law_title
law_sections = (
    matched_speeches[["law_number", "section_key", "date", "law_title"]]
    .drop_duplicates()
    .rename(columns={"date": "discussion_date"})
)

# convert to list of tuples for executemany, since it expects a list of tuples
rows = list(law_sections.itertuples(index=False, name=None))

# Insert the unique law sections into the law_sections table, ignoring duplicates
cur.executemany("""
    INSERT OR IGNORE INTO law_sections (law_number, section_key, discussion_date, law_title)
    VALUES (?, ?, ?, ?)
""", rows) # the ? placeholders will be replaced by the values from the tuples in rows

con.commit()

In [44]:
# Read the generated law_section_id so each speech can point to its respective section
law_map = pd.read_sql_query("""
    SELECT law_section_id, law_number, section_key
    FROM law_sections
""", con)

# Now we merge the law_section_id back into the matched_speeches dataframe
matched_speeches = matched_speeches.merge(
    law_map,
    on=["law_number", "section_key"],
    how="left"
)  

In [46]:
#law_map

In [47]:
# Now build the speeches table and insert it


# We take only the relevant columns for the speeches table
speeches = matched_speeches[
    ["law_section_id", "dipid", "speech_raw", "speech_clean"]
].copy()

# Keep only rows with actual cleaned text
speeches = speeches.loc[
    speeches["speech_clean"].notna() &
    speeches["speech_clean"].str.strip().ne("")
]

# convert to list of tuples for executemany 
rows = list(speeches.itertuples(index=False, name=None))

# Insert the speeches into the speeches table
cur.executemany("""
    INSERT INTO speeches (law_section_id, dipid, speech_raw, speech_clean)
    VALUES (?, ?, ?, ?)
""", rows)

con.commit()

In [49]:
# Checking that the speeches were inserted correctly

check_df = pd.read_sql_query("""
    SELECT
        s.speech_id,
        ls.law_number,
        ls.section_key,
        ls.discussion_date,
        s.dipid,
        d.name,
        d.lastname_1,
        d.lastname_2,
        s.speech_clean
    FROM speeches s
    JOIN law_sections ls
        ON s.law_section_id = ls.law_section_id
    LEFT JOIN deputies d
        ON s.dipid = d.dipid
    LIMIT 10
""", con)

check_df.head(20)

,speech_id,law_number,section_key,discussion_date,dipid,name,lastname_1,lastname_2,speech_clean
0,5808,21.472,1-5,2022-06-07,1015,Jorge,Brito,Hasbún,"presidente, demasiado importante ir ayuda dire..."
1,5809,21.472,1-5,2022-06-07,1126,Felipe,Donoso,Castro,"presidente, gustaría expresar deficitario proy..."
2,5810,21.472,1-5,2022-06-07,1161,Francisco,Pulgar,Castillo,"presidente, junto saludar energía, quiero deci..."
3,5811,21.472,1-5,2022-06-07,1019,José Miguel,Castro,Bascuñán,"presidente, malo, así simple: malo. aplaudir f..."
4,5812,21.472,1-5,2022-06-07,872,Jaime,Mulet,Martínez,"presidente, aprobaremos ley, dentro contexto c..."
5,5813,21.472,1-5,2022-06-07,1081,Francisco,Undurraga,Gazitúa,"presidente, intermedio saludo ministro. minist..."
6,5814,21.472,1-5,2022-06-07,1179,Alberto,Undurraga,Vicuña,"presidenta, verdad hace cargo urgencia, si apr..."
7,5815,21.472,1-5,2022-06-07,1050,Miguel,Mellado,Suazo,"presidenta, comisión hacienda conversamos toma..."
8,5816,21.472,1-5,2022-06-07,1110,Carlos,Bianchi,Chelech,"presidenta, siento absolutamente necesario dec..."
9,5817,21.472,1-5,2022-06-07,1017,Álvaro,Carter,Fernández,"presidenta, letra chica, bien chica. actual go..."


In [ ]:
final_match.to_csv(SAVE_RESULTS, index=False, encoding="utf-8-sig")

### Exploratory code

In [ ]:
# cur.execute("DELETE FROM speeches")
# cur.execute("DELETE FROM law_sections")
# con.commit()